In [1]:
!git clone https://github.com/bgmgncr/AI-Engineering-Evalution-Export-and-Optimization.git
%cd AI-Engineering-Evalution-Export-and-Optimization

Cloning into 'AI-Engineering-Evalution-Export-and-Optimization'...
/content/AI-Engineering-Evalution-Export-and-Optimization


In [29]:
%%writefile benchmark.py
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ultralytics import YOLO
import onnxruntime as ort


DEFAULT_IMAGES = [
    "data_samples/bus.jpg",
    "data_samples/zidane.jpg",
]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["pytorch", "onnx"], required=True)
    p.add_argument("--images", type=int, default=10)
    p.add_argument("--out", type=str, default="results/")
    p.add_argument("--weights", type=str, default="yolov8n.pt")  # for pytorch
    p.add_argument("--onnx_path", type=str, default="yolov8n.onnx")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--warmup", type=int, default=10)
    return p.parse_args()


def summarize_latency_ms(samples_ms):
    s = np.asarray(samples_ms, dtype=float)
    return {
        "mean_ms": float(s.mean()),
        "p50_ms": float(np.percentile(s, 50)),
        "p95_ms": float(np.percentile(s, 95)),
    }


def get_image_list(n):
    return (DEFAULT_IMAGES * ((n + len(DEFAULT_IMAGES) - 1) // len(DEFAULT_IMAGES)))[:n]


def run_ultralytics_backend(model, image_list, imgsz, warmup, device):
    # Warmup
    for _ in range(warmup):
        _ = model.predict(image_list[0], imgsz=imgsz, verbose=False, device=device)

    predictions = []
    times_ms = []

    for i, img in enumerate(image_list):
        t0 = time.perf_counter()
        res = model.predict(img, imgsz=imgsz, verbose=False, device=device)[0]
        # sync for accurate GPU timing (safe even if CPU)
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls = res.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(cls)):
                predictions.append(
                    {
                        "image_id": f"img_{i:04d}",
                        "class_id": int(cls[j]),
                        "score": float(conf[j]),
                        "bbox_xyxy": [float(x) for x in xyxy[j].tolist()],
                    }
                )

    return predictions, times_ms


def write_outputs(out_dir, backend, images, imgsz, predictions, times_ms):
    (out_dir / f"predictions_{backend}.json").write_text(json.dumps(predictions, indent=2))

    lat = summarize_latency_ms(times_ms)
    latency_df = pd.DataFrame([{
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        **lat
    }])
    latency_df.to_csv(out_dir / f"latency_{backend}.csv", index=False)

    # Placeholder metrics for now
    metrics_df = pd.DataFrame([{
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        "mAP50": None
    }])
    metrics_df.to_csv(out_dir / f"metrics_{backend}.csv", index=False)

    print("Wrote:", out_dir / f"predictions_{backend}.json")
    print("Wrote:", out_dir / f"latency_{backend}.csv")
    print("Wrote:", out_dir / f"metrics_{backend}.csv")



def main():
    args = parse_args()
    backend = args.backend
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_list = get_image_list(args.images)

    if args.backend == "pytorch":
        model = YOLO(args.weights)
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # Changed from 'cpu' to 0 for GPU
        )
        write_outputs(out_dir, "pytorch", args.images, args.imgsz, preds, times_ms)

    elif args.backend == "onnx":
        # Ensure CUDA provider exists
        providers = ort.get_available_providers()
        if "CUDAExecutionProvider" not in providers:
            raise RuntimeError(f"CUDAExecutionProvider not available. Providers: {providers}")

        model = YOLO(args.onnx_path)  # Ultralytics will run ONNX via onnxruntime
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # GPU
        )
        write_outputs(out_dir, "onnx", args.images, args.imgsz, preds, times_ms)


if __name__ == "__main__":
    main()

Overwriting benchmark.py


In [2]:
import os

repo_dir = "AI-Engineering-Evalution-Export-and-Optimization"
current_path = os.getcwd()

# Ensure we are in the correct directory (repo root)
if not current_path.endswith(repo_dir):
    %cd {repo_dir}

# List the contents of the results directory
!ls results/

[Errno 2] No such file or directory: 'AI-Engineering-Evalution-Export-and-Optimization'
/content
ls: cannot access 'results/': No such file or directory


In [3]:
import os

repo_dir = "AI-Engineering-Evalution-Export-and-Optimization"
current_path = os.getcwd()

# Ensure we are in the correct directory (repo root)
if not current_path.endswith(repo_dir):
    %cd {repo_dir}

# List the contents of the results directory to show the created CSV files
!ls results/

[Errno 2] No such file or directory: 'AI-Engineering-Evalution-Export-and-Optimization'
/content
ls: cannot access 'results/': No such file or directory


# Task
Benchmark YOLOv8n models using PyTorch and ONNX backends on GPU, saving the latency and metrics results to "/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/", and then summarize the generated results including latency metrics for GPU performance comparison.

## Mount Google Drive

### Subtask:
Mount the user's Google Drive to make it accessible within the Colab environment. This is a prerequisite for saving files directly to Drive.


**Reasoning**:
Mount Google Drive to make it accessible within the Colab environment for saving files to Drive.



In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Ensure Repository and Working Directory

### Subtask:
Ensure the 'AI-Engineering-Evalution-Export-and-Optimization' repository is cloned and set the current working directory to its root. This step will prevent 'file not found' errors encountered previously.


**Reasoning**:
To ensure a clean clone and prevent issues from previous attempts, I will first remove any existing repository directory and then clone the repository again.



In [5]:
get_ipython().system('rm -rf AI-Engineering-Evalution-Export-and-Optimization')
get_ipython().system('git clone https://github.com/bgmgncr/AI-Engineering-Evalution-Export-and-Optimization.git')

Cloning into 'AI-Engineering-Evalution-Export-and-Optimization'...


**Reasoning**:
The previous clone command indicated an empty repository, but this is a warning, not an error. The next step is to change the current working directory to the repository's root to ensure all subsequent commands are executed in the correct context, and then list its contents to verify the setup.



In [6]:
import os

repo_dir = 'AI-Engineering-Evalution-Export-and-Optimization'

# Change to the repository directory
%cd {repo_dir}

# List the contents of the current directory to verify
!ls

/content/AI-Engineering-Evalution-Export-and-Optimization


**Reasoning**:
The previous `git clone` command warned that the repository is empty. This means the `benchmark.py` script, which was written earlier, is not present in the currently cloned repository. To fix this, I need to recreate the `benchmark.py` file inside the current working directory, which is the root of the cloned repository.



In [7]:
%%writefile benchmark.py
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ultralytics import YOLO
import onnxruntime as ort


DEFAULT_IMAGES = [
    "data_samples/bus.jpg",
    "data_samples/zidane.jpg",
]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["pytorch", "onnx"], required=True)
    p.add_argument("--images", type=int, default=10)
    p.add_argument("--out", type=str, default="results/")
    p.add_argument("--weights", type=str, default="yolov8n.pt")  # for pytorch
    p.add_argument("--onnx_path", type=str, default="yolov8n.onnx")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--warmup", type=int, default=10)
    return p.parse_args()


def summarize_latency_ms(samples_ms):
    s = np.asarray(samples_ms, dtype=float)
    return {
        "mean_ms": float(s.mean()),
        "p50_ms": float(np.percentile(s, 50)),
        "p95_ms": float(np.percentile(s, 95)),
    }


def get_image_list(n):
    return (DEFAULT_IMAGES * ((n + len(DEFAULT_IMAGES) - 1) // len(DEFAULT_IMAGES)))[:n]


def run_ultralytics_backend(model, image_list, imgsz, warmup, device):
    # Warmup
    for _ in range(warmup):
        _ = model.predict(image_list[0], imgsz=imgz, verbose=False, device=device)

    predictions = []
    times_ms = []

    for i, img in enumerate(image_list):
        t0 = time.perf_counter()
        res = model.predict(img, imgsz=imgsz, verbose=False, device=device)[0]
        # sync for accurate GPU timing (safe even if CPU)
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls = res.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(cls)):
                predictions.append(
                    {
                        "image_id": f"img_{i:04d}",
                        "class_id": int(cls[j]),
                        "score": float(conf[j]),
                        "bbox_xyxy": [float(x) for x in xyxy[j].tolist()],
                    }
                )

    return predictions, times_ms


def write_outputs(out_dir, backend, images, imgsz, predictions, times_ms):
    (out_dir / f"predictions_{backend}.json").write_text(json.dumps(predictions, indent=2))

    lat = summarize_latency_ms(times_ms)
    latency_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        **lat
    }
    ])
    latency_df.to_csv(out_dir / f"latency_{backend}.csv", index=False)

    # Placeholder metrics for now
    metrics_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        "mAP50": None
    }
    ])
    metrics_df.to_csv(out_dir / f"metrics_{backend}.csv", index=False)

    print("Wrote:", out_dir / f"predictions_{backend}.json")
    print("Wrote:", out_dir / f"latency_{backend}.csv")
    print("Wrote:", out_dir / f"metrics_{backend}.csv")



def main():
    args = parse_args()
    backend = args.backend
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_list = get_image_list(args.images)

    if args.backend == "pytorch":
        model = YOLO(args.weights)
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # Changed from 'cpu' to 0 for GPU
        )
        write_outputs(out_dir, "pytorch", args.images, args.imgsz, preds, times_ms)

    elif args.backend == "onnx":
        # Ensure CUDA provider exists
        providers = ort.get_available_providers()
        if "CUDAExecutionProvider" not in providers:
            raise RuntimeError(f"CUDAExecutionProvider not available. Providers: {providers}")

        model = YOLO(args.onnx_path)  # Ultralytics will run ONNX via onnxruntime
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # GPU
        )
        write_outputs(out_dir, "onnx", args.images, args.imgsz, preds, times_ms)


if __name__ == "__main__":
    main()


Writing benchmark.py


**Reasoning**:
The next logical step after writing the `benchmark.py` script is to install the necessary Python packages for the script to run, specifically `ultralytics` which is used for YOLO models and `onnxruntime-gpu` for ONNX inference on GPU. This ensures all dependencies are met before attempting to run the benchmarking script.



In [8]:
get_ipython().system('pip install ultralytics onnxruntime-gpu')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.6/252.6 MB 3.5 MB/s eta 0:00:00


**Reasoning**:
The `benchmark.py` script requires sample images in a `data_samples` directory. This step will create the directory and download the specified images to ensure the script has the necessary input files to run without errors.



In [9]:
import os
import requests

# Create data_samples directory if it doesn't exist
data_samples_dir = "data_samples"
os.makedirs(data_samples_dir, exist_ok=True)

# Define image URLs and local paths
image_urls = {
    "bus.jpg": "https://ultralytics.com/images/bus.jpg",
    "zidane.jpg": "https://ultralytics.com/images/zidane.jpg"
}

# Download images
for filename, url in image_urls.items():
    filepath = os.path.join(data_samples_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        r = requests.get(url, stream=True)
        r.raise_for_status() # Raise an exception for HTTP errors
        with open(filepath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded {filename}")
    else:
        print(f"{filename} already exists, skipping download.")

print(f"Contents of {data_samples_dir}:")
!ls {data_samples_dir}

Downloaded bus.jpg
Downloaded zidane.jpg
Contents of data_samples:
bus.jpg  zidane.jpg


**Reasoning**:
Now that the `benchmark.py` script is available, dependencies are installed, and sample images are downloaded, the next step is to run the benchmarking for the PyTorch backend on GPU. The results will be saved to the specified Google Drive path, which needs to be created first.



In [10]:
import os

# Define the output directory in Google Drive
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/'

# Create the output directory if it doesn't exist
os.makedirs(output_dir_drive, exist_ok=True)
print(f"Ensured output directory exists: {output_dir_drive}")

# Run the benchmark for PyTorch backend on GPU
# The script is designed to use device=0 (GPU) when onnxruntime-gpu is installed
get_ipython().system(f'python benchmark.py --backend pytorch --out "{output_dir_drive}" --imgsz 640 --images 10')


Ensured output directory exists: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Traceback (most recent call last):
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 151, in <module>
    main()
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 124, in main
    preds, times_ms = run_ultralytics_backend(
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 47, in run_ultralytics_backend
    _ = model.predict(image_list[0], imgsz=imgz, verbose=False, device=device)
                    

**Reasoning**:
The previous execution failed due to a `NameError` where `imgz` was used instead of `imgsz` in the `run_ultralytics_backend` function within `benchmark.py`. I need to correct this typo in the script and then re-write the corrected `benchmark.py` file.



In [11]:
%%writefile benchmark.py
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ultralytics import YOLO
import onnxruntime as ort


DEFAULT_IMAGES = [
    "data_samples/bus.jpg",
    "data_samples/zidane.jpg",
]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["pytorch", "onnx"], required=True)
    p.add_argument("--images", type=int, default=10)
    p.add_argument("--out", type=str, default="results/")
    p.add_argument("--weights", type=str, default="yolov8n.pt")  # for pytorch
    p.add_argument("--onnx_path", type=str, default="yolov8n.onnx")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--warmup", type=int, default=10)
    return p.parse_args()


def summarize_latency_ms(samples_ms):
    s = np.asarray(samples_ms, dtype=float)
    return {
        "mean_ms": float(s.mean()),
        "p50_ms": float(np.percentile(s, 50)),
        "p95_ms": float(np.percentile(s, 95)),
    }


def get_image_list(n):
    return (DEFAULT_IMAGES * ((n + len(DEFAULT_IMAGES) - 1) // len(DEFAULT_IMAGES)))[:n]


def run_ultralytics_backend(model, image_list, imgsz, warmup, device):
    # Warmup
    for _ in range(warmup):
        _ = model.predict(image_list[0], imgsz=imgsz, verbose=False, device=device)

    predictions = []
    times_ms = []

    for i, img in enumerate(image_list):
        t0 = time.perf_counter()
        res = model.predict(img, imgsz=imgsz, verbose=False, device=device)[0]
        # sync for accurate GPU timing (safe even if CPU)
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls = res.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(cls)):
                predictions.append(
                    {
                        "image_id": f"img_{i:04d}",
                        "class_id": int(cls[j]),
                        "score": float(conf[j]),
                        "bbox_xyxy": [float(x) for x in xyxy[j].tolist()],
                    }
                )

    return predictions, times_ms


def write_outputs(out_dir, backend, images, imgsz, predictions, times_ms):
    (out_dir / f"predictions_{backend}.json").write_text(json.dumps(predictions, indent=2))

    lat = summarize_latency_ms(times_ms)
    latency_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        **lat
    }
    ])
    latency_df.to_csv(out_dir / f"latency_{backend}.csv", index=False)

    # Placeholder metrics for now
    metrics_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        "mAP50": None
    }
    ])
    metrics_df.to_csv(out_dir / f"metrics_{backend}.csv", index=False)

    print("Wrote:", out_dir / f"predictions_{backend}.json")
    print("Wrote:", out_dir / f"latency_{backend}.csv")
    print("Wrote:", out_dir / f"metrics_{backend}.csv")



def main():
    args = parse_args()
    backend = args.backend
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_list = get_image_list(args.images)

    if args.backend == "pytorch":
        model = YOLO(args.weights)
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # Changed from 'cpu' to 0 for GPU
        )
        write_outputs(out_dir, "pytorch", args.images, args.imgsz, preds, times_ms)

    elif args.backend == "onnx":
        # Ensure CUDA provider exists
        providers = ort.get_available_providers()
        if "CUDAExecutionProvider" not in providers:
            raise RuntimeError(f"CUDAExecutionProvider not available. Providers: {providers}")

        model = YOLO(args.onnx_path)  # Ultralytics will run ONNX via onnxruntime
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # GPU
        )
        write_outputs(out_dir, "onnx", args.images, args.imgsz, preds, times_ms)


if __name__ == "__main__":
    main()

Overwriting benchmark.py


**Reasoning**:
The `benchmark.py` script has been corrected. Now, I need to re-run the benchmarking for the PyTorch backend on GPU using the corrected script. This will generate the performance results and save them to the specified Google Drive path.



In [12]:
import os

# Define the output directory in Google Drive
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/'

# Create the output directory if it doesn't exist (already done in previous step, but harmless to repeat)
os.makedirs(output_dir_drive, exist_ok=True)
print(f"Ensured output directory exists: {output_dir_drive}")

# Run the benchmark for PyTorch backend on GPU
# The script is designed to use device=0 (GPU) when onnxruntime-gpu is installed
get_ipython().system(f'python benchmark.py --backend pytorch --out "{output_dir_drive}" --imgsz 640 --images 10')

Ensured output directory exists: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/
Wrote: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/predictions_pytorch.json
Wrote: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/latency_pytorch.csv
Wrote: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/metrics_pytorch.csv


**Reasoning**:
The PyTorch backend benchmarking has been completed. The next step is to perform the benchmarking for the ONNX backend using the same corrected `benchmark.py` script. The results will be saved to the same Google Drive path, alongside the PyTorch results.



In [13]:
import os

# Define the output directory in Google Drive
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/'

# Create the output directory if it doesn't exist (already done in previous step, but harmless to repeat)
os.makedirs(output_dir_drive, exist_ok=True)
print(f"Ensured output directory exists: {output_dir_drive}")

# Run the benchmark for ONNX backend on GPU
# The script is designed to use device=0 (GPU) when onnxruntime-gpu is installed
# Note: This will download yolov8n.onnx if not already present.
get_ipython().system(f'python benchmark.py --backend onnx --out "{output_dir_drive}" --imgsz 640 --images 10')

Ensured output directory exists: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/
Traceback (most recent call last):
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 151, in <module>
    main()
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 139, in main
    model = YOLO(args.onnx_path)  # Ultralytics will run ONNX via onnxruntime
            ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 288, in _load
    weights = checks.check_file(weights)  # runs in all cases, not redundant with above call
              ^^^^^^^

**Reasoning**:
The previous step failed because the `yolov8n.onnx` model was not found. I need to export the `yolov8n.pt` PyTorch model to ONNX format before it can be used for ONNX benchmarking. This will create the required `yolov8n.onnx` file in the current directory.



In [14]:
from ultralytics import YOLO

# Load the PyTorch YOLOv8n model
model = YOLO('yolov8n.pt')

# Export the model to ONNX format
# The output file will be yolov8n.onnx in the current directory
model.export(format='onnx', imgsz=640)

print("YOLOv8n model exported to yolov8n.onnx")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 184ms
Prepared 4 packages in 1.43s
Installed 4 packages in 388ms
 + colorama==0.4.6
 + onnx==1.20.1
 + onnxruntime==1.24.1
 + onnxslim==0.1.85

requirements: AutoUpdate success ✅ 2.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 4.9s, saved as 'yolov8n.onnx' (12.3 MB)

Export complete (5.4s)
Results saved to /content/AI-Engineering-Evalution-Export-and-Optimization
Predict:         yolo predict task=detect model=yolov8n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolov8n.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app
YOLOv8n model exported to yolov8n.onnx


**Reasoning**:
The `yolov8n.onnx` model has now been successfully exported. The next logical step is to re-run the benchmarking for the ONNX backend, as the required ONNX model file is now available.



In [15]:
import os

# Define the output directory in Google Drive
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/'

# Create the output directory if it doesn't exist (already done in previous step, but harmless to repeat)
os.makedirs(output_dir_drive, exist_ok=True)
print(f"Ensured output directory exists: {output_dir_drive}")

# Run the benchmark for ONNX backend on GPU
# The script is designed to use device=0 (GPU) when onnxruntime-gpu is installed
# Note: This will download yolov8n.onnx if not already present.
get_ipython().system(f'python benchmark.py --backend onnx --out "{output_dir_drive}" --imgsz 640 --images 10')

Ensured output directory exists: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/
Traceback (most recent call last):
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 151, in <module>
    main()
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 137, in main
    raise RuntimeError(f"CUDAExecutionProvider not available. Providers: {providers}")
RuntimeError: CUDAExecutionProvider not available. Providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


**Reasoning**:
The `RuntimeError: CUDAExecutionProvider not available` indicates that ONNX Runtime is not correctly utilizing the GPU, despite `onnxruntime-gpu` being installed. This often happens if the CUDA provider isn't explicitly prioritized or found. To fix this, I will modify the `benchmark.py` script to explicitly set `providers=['CUDAExecutionProvider']` when initializing the ONNX Runtime session, ensuring it attempts to use the GPU.



In [16]:
%%writefile benchmark.py
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ultralytics import YOLO
import onnxruntime as ort


DEFAULT_IMAGES = [
    "data_samples/bus.jpg",
    "data_samples/zidane.jpg",
]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["pytorch", "onnx"], required=True)
    p.add_argument("--images", type=int, default=10)
    p.add_argument("--out", type=str, default="results/")
    p.add_argument("--weights", type=str, default="yolov8n.pt")  # for pytorch
    p.add_argument("--onnx_path", type=str, default="yolov8n.onnx")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--warmup", type=int, default=10)
    return p.parse_args()


def summarize_latency_ms(samples_ms):
    s = np.asarray(samples_ms, dtype=float)
    return {
        "mean_ms": float(s.mean()),
        "p50_ms": float(np.percentile(s, 50)),
        "p95_ms": float(np.percentile(s, 95)),
    }


def get_image_list(n):
    return (DEFAULT_IMAGES * ((n + len(DEFAULT_IMAGES) - 1) // len(DEFAULT_IMAGES)))[:n]


def run_ultralytics_backend(model, image_list, imgsz, warmup, device):
    # Warmup
    for _ in range(warmup):
        _ = model.predict(image_list[0], imgsz=imgsz, verbose=False, device=device)

    predictions = []
    times_ms = []

    for i, img in enumerate(image_list):
        t0 = time.perf_counter()
        res = model.predict(img, imgsz=imgsz, verbose=False, device=device)[0]
        # sync for accurate GPU timing (safe even if CPU)
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls = res.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(cls)):
                predictions.append(
                    {
                        "image_id": f"img_{i:04d}",
                        "class_id": int(cls[j]),
                        "score": float(conf[j]),
                        "bbox_xyxy": [float(x) for x in xyxy[j].tolist()],
                    }
                )

    return predictions, times_ms


def write_outputs(out_dir, backend, images, imgsz, predictions, times_ms):
    (out_dir / f"predictions_{backend}.json").write_text(json.dumps(predictions, indent=2))

    lat = summarize_latency_ms(times_ms)
    latency_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        **lat
    }
    ])
    latency_df.to_csv(out_dir / f"latency_{backend}.csv", index=False)

    # Placeholder metrics for now
    metrics_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        "mAP50": None
    }
    ])
    metrics_df.to_csv(out_dir / f"metrics_{backend}.csv", index=False)

    print("Wrote:", out_dir / f"predictions_{backend}.json")
    print("Wrote:", out_dir / f"latency_{backend}.csv")
    print("Wrote:", out_dir / f"metrics_{backend}.csv")



def main():
    args = parse_args()
    backend = args.backend
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_list = get_image_list(args.images)

    if args.backend == "pytorch":
        model = YOLO(args.weights)
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # Changed from 'cpu' to 0 for GPU
        )
        write_outputs(out_dir, "pytorch", args.images, args.imgsz, preds, times_ms)

    elif args.backend == "onnx":
        # Explicitly set providers to include CUDAExecutionProvider
        providers = ['CUDAExecutionProvider']
        sess_options = ort.SessionOptions()
        model = YOLO(args.onnx_path, session=ort.InferenceSession(args.onnx_path, sess_options, providers=providers))  # Ultralytics will run ONNX via onnxruntime
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # GPU
        )
        write_outputs(out_dir, "onnx", args.images, args.imgsz, preds, times_ms)


if __name__ == "__main__":
    main()


Overwriting benchmark.py


**Reasoning**:
The `benchmark.py` script has been corrected to explicitly use the `CUDAExecutionProvider` for ONNX Runtime. Now, I need to re-run the benchmarking for the ONNX backend to ensure it utilizes the GPU and correctly generates the performance results, saving them to the specified Google Drive path.



In [17]:
import os

# Define the output directory in Google Drive
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/'

# Create the output directory if it doesn't exist (already done in previous step, but harmless to repeat)
os.makedirs(output_dir_drive, exist_ok=True)
print(f"Ensured output directory exists: {output_dir_drive}")

# Run the benchmark for ONNX backend on GPU
# The script is designed to use device=0 (GPU) when onnxruntime-gpu is installed
get_ipython().system(f'python benchmark.py --backend onnx --out "{output_dir_drive}" --imgsz 640 --images 10')

Ensured output directory exists: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
Traceback (most recent call last):
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 149, in <module>
    main()
  File "/content/AI-Engineering-Evalution-Export-and-Optimization/benchmark.py", line 137, in main
    model = YOLO(args.onnx_path, session=ort.InferenceSession(args.onnx_path, sess_options, providers=providers))  # Ultralytics will run ONNX via onnxruntime
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: YOLO.__init__() got an unexpected keyword argument 'session'


**Reasoning**:
The previous error `TypeError: YOLO.__init__() got an unexpected keyword argument 'session'` indicates that the `ultralytics.YOLO` constructor does not directly accept an `onnxruntime.InferenceSession` object. Instead, when loading an ONNX model, `YOLO` expects the path to the ONNX model and internally manages the ONNX Runtime session. I need to revert the change that explicitly passed a session object and allow `ultralytics` to handle the ONNX model loading by simply providing the ONNX path and device.



In [18]:
%%writefile benchmark.py
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ultralytics import YOLO
import onnxruntime as ort


DEFAULT_IMAGES = [
    "data_samples/bus.jpg",
    "data_samples/zidane.jpg",
]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["pytorch", "onnx"], required=True)
    p.add_argument("--images", type=int, default=10)
    p.add_argument("--out", type=str, default="results/")
    p.add_argument("--weights", type=str, default="yolov8n.pt")  # for pytorch
    p.add_argument("--onnx_path", type=str, default="yolov8n.onnx")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--warmup", type=int, default=10)
    return p.parse_args()


def summarize_latency_ms(samples_ms):
    s = np.asarray(samples_ms, dtype=float)
    return {
        "mean_ms": float(s.mean()),
        "p50_ms": float(np.percentile(s, 50)),
        "p95_ms": float(np.percentile(s, 95)),
    }


def get_image_list(n):
    return (DEFAULT_IMAGES * ((n + len(DEFAULT_IMAGES) - 1) // len(DEFAULT_IMAGES)))[:n]


def run_ultralytics_backend(model, image_list, imgsz, warmup, device):
    # Warmup
    for _ in range(warmup):
        _ = model.predict(image_list[0], imgsz=imgsz, verbose=False, device=device)

    predictions = []
    times_ms = []

    for i, img in enumerate(image_list):
        t0 = time.perf_counter()
        res = model.predict(img, imgsz=imgsz, verbose=False, device=device)[0]
        # sync for accurate GPU timing (safe even if CPU)
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls = res.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(cls)):
                predictions.append(
                    {
                        "image_id": f"img_{i:04d}",
                        "class_id": int(cls[j]),
                        "score": float(conf[j]),
                        "bbox_xyxy": [float(x) for x in xyxy[j].tolist()],
                    }
                )

    return predictions, times_ms


def write_outputs(out_dir, backend, images, imgsz, predictions, times_ms):
    (out_dir / f"predictions_{backend}.json").write_text(json.dumps(predictions, indent=2))

    lat = summarize_latency_ms(times_ms)
    latency_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        **lat
    }
    ])
    latency_df.to_csv(out_dir / f"latency_{backend}.csv", index=False)

    # Placeholder metrics for now
    metrics_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        "mAP50": None
    }
    ])
    metrics_df.to_csv(out_dir / f"metrics_{backend}.csv", index=False)

    print("Wrote:", out_dir / f"predictions_{backend}.json")
    print("Wrote:", out_dir / f"latency_{backend}.csv")
    print("Wrote:", out_dir / f"metrics_{backend}.csv")



def main():
    args = parse_args()
    backend = args.backend
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_list = get_image_list(args.images)

    if args.backend == "pytorch":
        model = YOLO(args.weights)
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # Changed from 'cpu' to 0 for GPU
        )
        write_outputs(out_dir, "pytorch", args.images, args.imgsz, preds, times_ms)

    elif args.backend == "onnx":
        # Revert changes: Ultralytics YOLO constructor handles ONNX session internally
        # and implicitly uses CUDA if onnxruntime-gpu is installed and CUDA is available.
        # Removing explicit session passing that caused TypeError.
        model = YOLO(args.onnx_path)  # Ultralytics will run ONNX via onnxruntime
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # GPU
        )
        write_outputs(out_dir, "onnx", args.images, args.imgsz, preds, times_ms)


if __name__ == "__main__":
    main()


Overwriting benchmark.py


**Reasoning**:
The `benchmark.py` script has been corrected to properly initialize the `YOLO` model for the ONNX backend. Now, I need to re-run the benchmarking for the ONNX backend to ensure it utilizes the GPU and correctly generates the performance results, saving them to the specified Google Drive path.



In [19]:
import os

# Define the output directory in Google Drive
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/'

# Create the output directory if it doesn't exist (already done in previous step, but harmless to repeat)
os.makedirs(output_dir_drive, exist_ok=True)
print(f"Ensured output directory exists: {output_dir_drive}")

# Run the benchmark for ONNX backend on GPU
# The script is designed to use device=0 (GPU) when onnxruntime-gpu is installed
get_ipython().system(f'python benchmark.py --backend onnx --out "{output_dir_drive}" --imgsz 640 --images 10')

Ensured output directory exists: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading yolov8n.onnx for ONNX Runtime inference...
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.24.1 with CPUExecutionProvider
Wrote: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/predictions_onnx.json
Wrote: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/latency_onnx.csv
Wrote: /content/drive/MyDrive/Colab Notebooks/AI-Engineering(Evalution,Export and Optimization)/results/metrics_onnx.csv


**Reasoning**:
The previous ONNX benchmark run indicates that CUDA was not used, falling back to CPU. This means `onnxruntime-gpu` is not correctly configured or utilized. To investigate and potentially fix this, I need to list the available ONNX Runtime execution providers to understand the current environment setup and verify if CUDA is indeed recognized.



In [20]:
import onnxruntime as ort

print("Available ONNX Runtime providers:", ort.get_available_providers())

Available ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


## Retry ONNX Benchmark on GPU after Runtime Restart

### Subtask:
The previous attempt to run the ONNX benchmark on GPU failed because `CUDAExecutionProvider` was not available to ONNX Runtime. This subtask involves restarting the Colab runtime to refresh the environment, re-executing all necessary setup steps (mounting Drive, cloning repo, installing dependencies, preparing data), and then re-running the ONNX benchmark to ensure GPU utilization.


**Reasoning**:
As instructed, after a runtime restart, the first step is to re-mount Google Drive to make it accessible for saving results. The previous output shows it was already mounted, but since a restart is implied, this step ensures it's correctly re-mounted.



In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Reasoning**:
Following the instructions after a runtime restart, the next step is to ensure the repository is cleanly cloned. I will first remove any existing directory and then clone the repository again to ensure a fresh state, which is crucial as the previous clone command warned about an empty repository.



In [22]:
get_ipython().system('rm -rf AI-Engineering-Evalution-Export-and-Optimization')
get_ipython().system('git clone https://github.com/bgmgncr/AI-Engineering-Evalution-Export-and-Optimization.git')

Cloning into 'AI-Engineering-Evalution-Export-and-Optimization'...


**Reasoning**:
The previous clone command indicated an empty repository, but this is a warning, not an error. The next step is to change the current working directory to the repository's root to ensure all subsequent commands are executed in the correct context, and then list its contents to verify the setup.



In [23]:
import os

repo_dir = 'AI-Engineering-Evalution-Export-and-Optimization'

# Change to the repository directory
%cd {repo_dir}

# List the contents of the current directory to verify
!ls

/content/AI-Engineering-Evalution-Export-and-Optimization/AI-Engineering-Evalution-Export-and-Optimization


**Reasoning**:
The previous `cd` command resulted in a nested directory structure. I need to navigate back one level to the actual root of the cloned repository (`/content/AI-Engineering-Evalution-Export-and-Optimization`) to ensure that `benchmark.py` and other files are created in the correct location for subsequent steps.



In [24]:
import os

# Navigate up one directory level to the repository root
%cd ..

# List the contents of the current directory to verify
!ls

/content/AI-Engineering-Evalution-Export-and-Optimization
AI-Engineering-Evalution-Export-and-Optimization  data_samples	yolov8n.pt
benchmark.py					  yolov8n.onnx


**Reasoning**:
The previous steps ensure the repository is correctly set up and the working directory is correct. Now, I need to re-write the `benchmark.py` script to include the necessary corrections (e.g., `imgsz` typo fix and correct ONNX model loading) as performed previously, ensuring the latest working version of the script is available in the current directory.



In [25]:
%%writefile benchmark.py
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from ultralytics import YOLO
import onnxruntime as ort


DEFAULT_IMAGES = [
    "data_samples/bus.jpg",
    "data_samples/zidane.jpg",
]


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--backend", choices=["pytorch", "onnx"], required=True)
    p.add_argument("--images", type=int, default=10)
    p.add_argument("--out", type=str, default="results/")
    p.add_argument("--weights", type=str, default="yolov8n.pt")  # for pytorch
    p.add_argument("--onnx_path", type=str, default="yolov8n.onnx")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--warmup", type=int, default=10)
    return p.parse_args()


def summarize_latency_ms(samples_ms):
    s = np.asarray(samples_ms, dtype=float)
    return {
        "mean_ms": float(s.mean()),
        "p50_ms": float(np.percentile(s, 50)),
        "p95_ms": float(np.percentile(s, 95)),
    }


def get_image_list(n):
    return (DEFAULT_IMAGES * ((n + len(DEFAULT_IMAGES) - 1) // len(DEFAULT_IMAGES)))[:n]


def run_ultralytics_backend(model, image_list, imgsz, warmup, device):
    # Warmup
    for _ in range(warmup):
        _ = model.predict(image_list[0], imgsz=imgsz, verbose=False, device=device)

    predictions = []
    times_ms = []

    for i, img in enumerate(image_list):
        t0 = time.perf_counter()
        res = model.predict(img, imgsz=imgsz, verbose=False, device=device)[0]
        # sync for accurate GPU timing (safe even if CPU)
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:
            pass
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls = res.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(cls)):
                predictions.append(
                    {
                        "image_id": f"img_{i:04d}",
                        "class_id": int(cls[j]),
                        "score": float(conf[j]),
                        "bbox_xyxy": [float(x) for x in xyxy[j].tolist()],
                    }
                )

    return predictions, times_ms


def write_outputs(out_dir, backend, images, imgsz, predictions, times_ms):
    (out_dir / f"predictions_{backend}.json").write_text(json.dumps(predictions, indent=2))

    lat = summarize_latency_ms(times_ms)
    latency_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        **lat
    }
    ])
    latency_df.to_csv(out_dir / f"latency_{backend}.csv", index=False)

    # Placeholder metrics for now
    metrics_df = pd.DataFrame([
        {
        "backend": backend,
        "images": images,
        "imgsz": imgsz,
        "mAP50": None
    }
    ])
    metrics_df.to_csv(out_dir / f"metrics_{backend}.csv", index=False)

    print("Wrote:", out_dir / f"predictions_{backend}.json")
    print("Wrote:", out_dir / f"latency_{backend}.csv")
    print("Wrote:", out_dir / f"metrics_{backend}.csv")


def main():
    args = parse_args()
    backend = args.backend
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    image_list = get_image_list(args.images)

    if args.backend == "pytorch":
        model = YOLO(args.weights)
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # Changed from 'cpu' to 0 for GPU
        )
        write_outputs(out_dir, "pytorch", args.images, args.imgsz, preds, times_ms)

    elif args.backend == "onnx":
        # Revert changes: Ultralytics YOLO constructor handles ONNX session internally
        # and implicitly uses CUDA if onnxruntime-gpu is installed and CUDA is available.
        # Removing explicit session passing that caused TypeError.
        model = YOLO(args.onnx_path)  # Ultralytics will run ONNX via onnxruntime
        preds, times_ms = run_ultralytics_backend(
            model=model,
            image_list=image_list,
            imgsz=args.imgsz,
            warmup=args.warmup,
            device=0,  # GPU
        )
        write_outputs(out_dir, "onnx", args.images, args.imgsz, preds, times_ms)


if __name__ == "__main__":
    main()

Overwriting benchmark.py


In [3]:
!yolo val model=yolov8n.pt data=VOC.yaml split=test imgsz=640 device=0

/bin/bash: line 1: yolo: command not found
